Unified Layer 1 (Input Safety) Dataset Construction

**Purpose.** This notebook combines two independently-built
**AAAC** (Algerian Arabic Adversarial Corpus) and **OSACT Layer1** — into a single
unified binary-classification resource (`safe` vs. `adversarial`) for training and
evaluating the AMAN-LLM Layer 1 Input-Safety classifier.

## 0. Configuration & Imports

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 80)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.unicode_minus"] = False

# Paths (edit DATA_DIR if running outside this environment)
DATA_DIR   = Path("/content/drive/MyDrive/Datasets")
OUTPUT_DIR = Path("/content/drive/MyDrive/Datasets/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

AAAC_FILES = {
    "train":      DATA_DIR / "AAAC" / "train_aaac.csv",
    "validation": DATA_DIR / "AAAC" / "val_aaac.csv",
    "test":       DATA_DIR / "AAAC" / "test_aaac.csv",
}
OSACT_FILES = {
    "train":      DATA_DIR / "OSACT" / "train_osact.csv",
    "validation": DATA_DIR / "OSACT" / "dev_osact.csv",
    "test":       DATA_DIR / "OSACT" / "test_osact.csv",
}

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


## PART 1 — Load & Verify Source Datasets

Each file is loaded exactly as-is. We verify that the expected columns are present
and print the shape of every split so any loading problem is caught immediately.


In [ ]:

AAAC_REQUIRED_COLS  = ["id", "text", "language_variant", "category", "tool"]
OSACT_REQUIRED_COLS = ["tweet_id", "text", "text_clean", "layer1_label", "source"]


def load_and_verify(path: Path, required_cols: list, name: str) -> pd.DataFrame:
    '''Load a CSV and assert the required schema is present. Never mutates the file on disk.'''
    if not path.exists():
        raise FileNotFoundError(f"{name}: expected file not found at {path}")
    df = pd.read_csv(path)
    missing = set(required_cols) - set(df.columns)
    if missing:
        raise ValueError(f"{name}: missing required column(s) {missing}")
    print(f"[OK] {name:22s} shape={df.shape!s:12s} file={path.name}")
    return df


print("Loading AAAC splits (official, frozen)")
print("-" * 60)
aaac_raw = {split: load_and_verify(p, AAAC_REQUIRED_COLS, f"AAAC {split}") for split, p in AAAC_FILES.items()}

print()
print("Loading OSACT Layer1 splits (official, frozen)")
print("-" * 60)
osact_raw = {split: load_and_verify(p, OSACT_REQUIRED_COLS, f"OSACT {split}") for split, p in OSACT_FILES.items()}


In [ ]:

# Quick sanity peek at each split (first 2 rows) — read-only inspection
for split in ["train", "validation", "test"]:
    print(f"\n--- AAAC {split} ---")
    display(aaac_raw[split].head(2))
for split in ["train", "validation", "test"]:
    print(f"\n--- OSACT {split} ---")
    display(osact_raw[split].head(2))


## PART 2 — Harmonize Schema

We map both datasets onto one unified table with columns:

In [ ]:

def harmonize_aaac(df: pd.DataFrame, split: str) -> pd.DataFrame:
    out = pd.DataFrame({
        "id":                df["id"].astype(str),
        "text":              df["text"],
        "language_variant":  df["language_variant"],
        "source":            "AAAC",
        "original_label":    df["category"],
        "dataset_split":     split,
        "is_adversarial_raw": (df["category"] != "safe").astype(int),
        "tool":              df["tool"],
        "text_clean":        np.nan,
    })
    return out


def harmonize_osact(df: pd.DataFrame, split: str) -> pd.DataFrame:
    out = pd.DataFrame({
        "id":                df["tweet_id"].astype(str),
        "text":              df["text"],
        "language_variant":  "osact_msa",
        "source":            "OSACT",
        "original_label":    df["layer1_label"],
        "dataset_split":     split,
        "is_adversarial_raw": df["layer1_label"].map({"safe": 0, "hate_speech": 1}),
        "tool":              np.nan,
        "text_clean":        df["text_clean"],
    })
    return out


harmonized_parts = (
    [harmonize_aaac(aaac_raw[s], s) for s in ["train", "validation", "test"]] +
    [harmonize_osact(osact_raw[s], s) for s in ["train", "validation", "test"]]
)
unified = pd.concat(harmonized_parts, ignore_index=True)
print("Unified (pre-label-mapping) shape:", unified.shape)


## PART 3 — Create Binary Layer 1 Labels

In [ ]:

AAAC_BINARY_MAP  = {0: "safe", 1: "adversarial", False: "safe", True: "adversarial"}
OSACT_BINARY_MAP = {"safe": "safe", "hate_speech": "adversarial"}

is_aaac  = unified["source"] == "AAAC"
is_osact = unified["source"] == "OSACT"

unified.loc[is_aaac, "binary_label"]  = unified.loc[is_aaac, "is_adversarial_raw"].map(AAAC_BINARY_MAP)
unified.loc[is_osact, "binary_label"] = unified.loc[is_osact, "original_label"].map(OSACT_BINARY_MAP)

FINAL_COLS = ["id", "text", "language_variant", "source", "binary_label",
              "original_label", "dataset_split", "is_adversarial_raw", "tool", "text_clean"]
unified = unified[FINAL_COLS].reset_index(drop=True)

print("Label mapping applied. binary_label distribution:")
print(unified["binary_label"].value_counts(dropna=False))


In [ ]:

label_map_table = pd.DataFrame([
    ("AAAC",  "category = safe", "safe"),
    ("AAAC",  "category != safe",  "adversarial"),
    ("OSACT", "layer1_label = safe",        "safe"),
    ("OSACT", "layer1_label = hate_speech", "adversarial"),
    ("OSACT", "layer1_label = <missing>  (blind test set)", "NaN (label withheld by source dataset)"),
], columns=["source", "original_label_rule", "binary_label"])
display(label_map_table)


## PART 4 — Dataset Validation

We check, and **report rather than silently fix**, four properties:

1. Missing values (per column)
2. Duplicated IDs
3. Invalid / unmapped labels
4. Invalid split values

In [ ]:

def validate_dataset(df: pd.DataFrame) -> dict:
    report = {}

    # 1. Missing values
    missing_counts = df.isna().sum()
    report["missing_values"] = missing_counts[missing_counts > 0].to_dict()

    # Split out the known/expected OSACT blind-test-label case
    blind_test_mask = (df["source"] == "OSACT") & (df["dataset_split"] == "test")
    report["expected_missing_labels_osact_blind_test"] = int(
        df.loc[blind_test_mask, "binary_label"].isna().sum()
    )
    unexpected_missing_labels = df.loc[~blind_test_mask, "binary_label"].isna().sum()
    report["unexpected_missing_labels"] = int(unexpected_missing_labels)
    if unexpected_missing_labels:
        report["unexpected_missing_label_rows"] = df.loc[
            (~blind_test_mask) & (df["binary_label"].isna()), ["id", "source", "dataset_split"]
        ].to_dict("records")

    # 2. Duplicated IDs (composite key: id + dataset_split) 
    dup_composite = df.duplicated(subset=["id", "dataset_split"]).sum()
    report["duplicated_id_split_pairs"] = int(dup_composite)
    dup_raw_id_global = df.duplicated(subset=["id"]).sum()
    report["duplicated_raw_id_global_note"] = (
        f"{dup_raw_id_global} raw `id` values repeat across splits "
        f"(expected/by-design for AAAC template ids; harmless once split is considered)"
    )

    # 3. Invalid labels
    valid_labels = {"safe", "adversarial"}
    invalid_labels = df.loc[~df["binary_label"].isin(valid_labels) & df["binary_label"].notna()]
    report["invalid_binary_labels"] = int(len(invalid_labels))

    # 4. Invalid splits
    valid_splits = {"train", "validation", "test"}
    invalid_splits = df.loc[~df["dataset_split"].isin(valid_splits)]
    report["invalid_splits"] = int(len(invalid_splits))

    return report


validation_report = validate_dataset(unified)

print("=" * 60)
print("VALIDATION REPORT")
print("=" * 60)
for k, v in validation_report.items():
    print(f"{k}: {v}")


## PART 4b — Resolve the Unexpected Missing Label

In [ ]:

unexpected_rows = unified.loc[
    unified["binary_label"].isna()
    & ~((unified["source"] == "OSACT") & (unified["dataset_split"] == "test"))
]
print(f"Unexpected-missing-label rows found: {len(unexpected_rows)}")
display(unexpected_rows[["id", "source", "dataset_split", "text"]])

if len(unexpected_rows):
    unified = unified.drop(index=unexpected_rows.index).reset_index(drop=True)
    print(f"Dropped {len(unexpected_rows)} row(s) with an unresolvable missing label "
          f"(raw source gap, not the OSACT blind test). New unified shape: {unified.shape}")

# Re-run validation to confirm the fix stuck
validation_report = validate_dataset(unified)
assert validation_report["unexpected_missing_labels"] == 0, (
    "Still have unexplained missing labels -- investigate before continuing."
)
print("\nRe-validated: unexpected_missing_labels =", validation_report["unexpected_missing_labels"])


## PART 5 — Cross-Dataset Duplicate Detection


In [ ]:

ARABIC_DIACRITICS = re.compile(r'[\u064B-\u065F\u0670\u06D6-\u06ED]')

def normalize_arabic(text) -> str:
    '''Analysis-only normalization for duplicate detection. Does not touch the exported `text` column.'''
    if pd.isna(text):
        return ""
    t = str(text)
    t = re.sub(r'http\S+|www\.\S+', ' ', t)
    t = re.sub(r'@USER|@\w+', ' ', t)
    t = re.sub(r'<LF>', ' ', t)
    t = ARABIC_DIACRITICS.sub('', t)
    t = re.sub('\u0640', '', t)          # tatweel
    t = re.sub('[إأآا]', 'ا', t)          # unify alef forms
    t = re.sub('[ى]', 'ي', t)             # unify ya / alef maksura
    t = re.sub('[ة]', 'ه', t)             # unify ta marbuta
    t = t.lower()
    t = re.sub(r'[^\w\s\u0600-\u06FF]', ' ', t)
    t = re.sub(r'\s+', ' ', t).strip()
    return t


unified["text_norm"] = unified["text"].apply(normalize_arabic)

# --- 1. Exact duplicates ------------------------------------------------
exact_dupe_mask = unified.duplicated(subset="text", keep=False) & (unified["text"].str.len() > 0)
exact_dupes = unified.loc[exact_dupe_mask].copy()
exact_dupes["dup_type"] = "exact_text"

# --- 2. Normalized duplicates (excluding those already exact) -----------
norm_dupe_mask = unified.duplicated(subset="text_norm", keep=False) & (unified["text_norm"].str.len() > 0)
norm_dupes = unified.loc[norm_dupe_mask & ~exact_dupe_mask].copy()
norm_dupes["dup_type"] = "normalized_text"

print(f"Exact duplicate rows:      {len(exact_dupes)}")
print(f"Normalized-only duplicate rows: {len(norm_dupes)}")


In [ ]:

# --- 3. Near duplicates (TF-IDF char n-grams + cosine kNN) ---------------
vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 4), min_df=2)
X = vectorizer.fit_transform(unified["text_norm"])

nn = NearestNeighbors(n_neighbors=2, metric="cosine", algorithm="brute")
nn.fit(X)
distances, neighbor_idx = nn.kneighbors(X)

cosine_sim_to_nearest = 1 - distances[:, 1]
nearest_partner_idx = neighbor_idx[:, 1]

NEAR_DUP_THRESHOLD = 0.85
near_dup_mask = (cosine_sim_to_nearest >= NEAR_DUP_THRESHOLD) & (~exact_dupe_mask.values) & (~norm_dupe_mask.values)

near_dupes = unified.loc[near_dup_mask].copy()
near_dupes["dup_type"] = "near_duplicate"
near_dupes["similarity_score"] = cosine_sim_to_nearest[near_dup_mask]
near_dupes["nearest_partner_id"] = unified.iloc[nearest_partner_idx[near_dup_mask]]["id"].values
near_dupes["nearest_partner_source"] = unified.iloc[nearest_partner_idx[near_dup_mask]]["source"].values

print(f"Near-duplicate rows (cosine >= {NEAR_DUP_THRESHOLD}): {len(near_dupes)}")


In [ ]:

# Assemble duplicates report
report_cols = ["id", "source", "dataset_split", "text", "binary_label", "dup_type"]

dup_report_frames = []
if len(exact_dupes):
    d = exact_dupes[report_cols].copy()
    d["similarity_score"] = 1.0
    d["nearest_partner_id"] = np.nan
    d["nearest_partner_source"] = np.nan
    dup_report_frames.append(d)
if len(norm_dupes):
    d = norm_dupes[report_cols].copy()
    d["similarity_score"] = 1.0
    d["nearest_partner_id"] = np.nan
    d["nearest_partner_source"] = np.nan
    dup_report_frames.append(d)
if len(near_dupes):
    dup_report_frames.append(near_dupes[report_cols + ["similarity_score", "nearest_partner_id", "nearest_partner_source"]])

duplicates_report = (
    pd.concat(dup_report_frames, ignore_index=True)
    if dup_report_frames else pd.DataFrame(columns=report_cols + ["similarity_score", "nearest_partner_id", "nearest_partner_source"])
)
duplicates_report = duplicates_report.sort_values(["dup_type", "source", "id"]).reset_index(drop=True)

duplicates_report.to_csv(OUTPUT_DIR / "duplicates_report.csv", index=False, encoding="utf-8")
print(f"duplicates_report.csv written: {len(duplicates_report)} flagged rows "
      f"({len(duplicates_report) / len(unified):.2%} of the unified dataset)")
print(duplicates_report["dup_type"].value_counts())


In [ ]:

# Cross-dataset overlap specifically (AAAC text appearing verbatim in OSACT or vice versa)
cross_source_exact = (
    exact_dupes.groupby("text")["source"]
    .apply(lambda s: s.nunique() > 1)
)
n_cross_source_groups = int(cross_source_exact.sum())
print(f"Exact-duplicate text groups that SPAN both AAAC and OSACT: {n_cross_source_groups}")


## PART 5b — Fix: Reserve Real OSACT-Domain Rows for the Unified Test Set

In [ ]:

OSACT_TEST_HOLDOUT_FRAC = 0.10

# Every row starts out "official" (i.e. exactly the split it shipped with)
unified["test_provenance"] = "official"

# Eligible pool: only OSACT rows currently sitting in dataset_split == "train".
# (OSACT validation is off-limits -- already used as layer1_validation.csv, see PART 4 note.)
osact_train_mask = (unified["source"] == "OSACT") & (unified["dataset_split"] == "train")
osact_train_idx  = unified.index[osact_train_mask]

_, holdout_idx = train_test_split(
    osact_train_idx,
    test_size=OSACT_TEST_HOLDOUT_FRAC,
    stratify=unified.loc[osact_train_idx, "binary_label"],
    random_state=RANDOM_SEED,
)

unified.loc[holdout_idx, "dataset_split"]    = "test"
unified.loc[holdout_idx, "test_provenance"]  = "osact_train_holdout_supplement"

print(f"OSACT train pool eligible for holdout: {len(osact_train_idx)}")
print(f"Rows moved OSACT train -> test:         {len(holdout_idx)} "
      f"({len(holdout_idx) / len(osact_train_idx):.1%})")
print()
print("Unified test composition AFTER the fix (source x provenance):")
print(unified.loc[unified['dataset_split'] == 'test'].groupby(['source', 'test_provenance']).size())
print()
print("Unified test composition AFTER the fix (source x binary_label, non-null only):")
print(
    unified.loc[unified['dataset_split'] == 'test']
           .groupby(['source', 'binary_label'], dropna=False)
           .size()
)

# Safety net: the holdout rows must have carried a real label (not NaN) -- if this trips,
# something upstream (e.g. the OSACT_BINARY_MAP) silently dropped labels again.
assert unified.loc[holdout_idx, "binary_label"].notna().all(), (
    "Some OSACT holdout rows moved into test have a NaN binary_label -- "
    "investigate before exporting."
)


## PART 6 — Dataset Statistics

Publication-ready summary tables and figures: overall composition, per-source,
per-split, per-label, per-language-variant breakdowns, token-length statistics,
and vocabulary size.


In [ ]:

def whitespace_tokenize(text) -> list:
    '''Simple whitespace tokenizer used ONLY for descriptive statistics (not for modeling).'''
    if pd.isna(text):
        return []
    return str(text).split()

unified["token_count"] = unified["text"].apply(lambda t: len(whitespace_tokenize(t)))

total_samples = len(unified)
print(f"Total unified samples: {total_samples}")


In [ ]:

# Table 1: samples per source 
tbl_source = unified.groupby("source").size().rename("count").to_frame()
tbl_source["percent"] = (tbl_source["count"] / total_samples * 100).round(2)
print("Table 1 — Samples per source")
display(tbl_source)

# Table 2: samples per split 
tbl_split = unified.groupby("dataset_split").size().rename("count").to_frame()
tbl_split["percent"] = (tbl_split["count"] / total_samples * 100).round(2)
tbl_split = tbl_split.reindex(["train", "validation", "test"])
print("\nTable 2 — Samples per split")
display(tbl_split)

# Table 3: samples per source x split 
tbl_source_split = unified.groupby(["source", "dataset_split"]).size().unstack("dataset_split")
tbl_source_split = tbl_source_split[["train", "validation", "test"]]
print("\nTable 3 — Samples per source x split")
display(tbl_source_split)

# Table 4: samples per binary label
tbl_label = unified.groupby("binary_label", dropna=False).size().rename("count").to_frame()
tbl_label["percent"] = (tbl_label["count"] / total_samples * 100).round(2)
print("\nTable 4 — Samples per binary label")
display(tbl_label)

# Table 5: samples per language variant
tbl_lang = unified.groupby("language_variant").size().rename("count").to_frame()
tbl_lang["percent"] = (tbl_lang["count"] / total_samples * 100).round(2)
print("\nTable 5 — Samples per language variant")
display(tbl_lang)

In [ ]:

# Table 6: token length statistics 
tbl_tokens = unified.groupby("source")["token_count"].agg(["mean", "median", "std", "min", "max"]).round(2)
tbl_tokens.loc["overall"] = unified["token_count"].agg(["mean", "median", "std", "min", "max"]).round(2)
print("Table 6 — Token length statistics (whitespace tokenization, descriptive only)")
display(tbl_tokens)

# Vocabulary size 
def vocab_size(texts: pd.Series) -> int:
    vocab = set()
    for t in texts.dropna():
        vocab.update(whitespace_tokenize(t))
    return len(vocab)

vocab_overall = vocab_size(unified["text"])
vocab_aaac = vocab_size(unified.loc[unified["source"] == "AAAC", "text"])
vocab_osact = vocab_size(unified.loc[unified["source"] == "OSACT", "text"])

print(f"\nVocabulary size — overall: {vocab_overall:,} | AAAC: {vocab_aaac:,} | OSACT: {vocab_osact:,}")


In [ ]:

# Figures 
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

tbl_source_split.plot(kind="bar", stacked=True, ax=axes[0], color=["#4C72B0", "#DD8452", "#55A868"])
axes[0].set_title("Samples per Source × Split")
axes[0].set_ylabel("count")
axes[0].legend(title="split")

unified.groupby(["source", "binary_label"], dropna=False).size().unstack("binary_label").plot(
    kind="bar", ax=axes[1], color=["#C44E52", "#4C72B0"]
)
axes[1].set_title("Binary Label Distribution per Source")
axes[1].set_ylabel("count")

unified["token_count"].clip(upper=unified["token_count"].quantile(0.99)).hist(
    bins=40, ax=axes[2], color="#4C72B0", edgecolor="white"
)
axes[2].set_title("Token Length Distribution\n(clipped at 99th percentile)")
axes[2].set_xlabel("token count")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "layer1_dataset_statistics.png", dpi=150, bbox_inches="tight")
plt.show()


## PART 7 — Dataset Balance Analysis

We examine class balance at three levels — AAAC alone, OSACT alone, and the
unified dataset — and give a scientifically-justified recommendation on
class weighting vs. resampling vs. no balancing.


In [ ]:

def balance_summary(df: pd.DataFrame, name: str) -> pd.DataFrame:
    counts = df["binary_label"].value_counts(dropna=False)
    total = counts.sum()
    ratio = (counts.get("safe", 0) / counts.get("adversarial", 1)) if counts.get("adversarial", 0) else np.nan
    summary = pd.DataFrame({
        "count": counts,
        "percent": (counts / total * 100).round(2),
    })
    summary.loc["safe:adversarial ratio"] = [round(ratio, 2), np.nan]
    print(f"--- {name} ---")
    display(summary)
    return summary

aaac_only  = unified[unified["source"] == "AAAC"]
osact_only = unified[unified["source"] == "OSACT"]

_ = balance_summary(aaac_only, "AAAC only (all splits)")
_ = balance_summary(osact_only, "OSACT only (all splits, excludes blind test)")
_ = balance_summary(unified, "Unified dataset (all splits, excludes blind test)")


In [ ]:

# Per-split balance, since imbalance that varies by split matters for eval design
tbl_balance_by_split = (
    unified.groupby(["dataset_split", "binary_label"], dropna=False)
    .size()
    .unstack("binary_label")
    .reindex(["train", "validation", "test"])
)
tbl_balance_by_split["adversarial_pct"] = (
    tbl_balance_by_split["adversarial"] / tbl_balance_by_split.sum(axis=1) * 100
).round(2)
print("Adversarial-class prevalence per split:")
display(tbl_balance_by_split)


## PART 8 — Export

The unified dataset is written out per the **original, official splits**, augmented only in `test` by the stratified OSACT-train holdout carved in **Part 5b** (fix for bug #7 — see `test_provenance` column) as UTF-8 CSV. No AAAC split changes; no rows are duplicated or discarded, only a subset of OSACT `train` rows change their `dataset_split` label.


In [ ]:

EXPORT_COLS = ["id", "text", "language_variant", "source", "binary_label",
               "original_label", "dataset_split", "is_adversarial_raw", "tool", "text_clean",
               "test_provenance"]

split_filenames = {
    "train":      "layer1_train.csv",
    "validation": "layer1_validation.csv",
    "test":       "layer1_test.csv",
}

for split, filename in split_filenames.items():
    subset = unified.loc[unified["dataset_split"] == split, EXPORT_COLS]
    out_path = OUTPUT_DIR / filename
    subset.to_csv(out_path, index=False, encoding="utf-8")
    print(f"[WRITTEN] {filename:24s} rows={len(subset):6d}  ->  {out_path}")


## PART 9 — Research Report (`Layer1_unified_report.csv`)

In [ ]:

report_rows = []

def add_row(section, metric, value):
    report_rows.append({"section": section, "metric": metric, "value": value})

# Dataset composition 
add_row("composition", "total_samples", total_samples)
for src, cnt in tbl_source["count"].items():
    add_row("composition", f"samples_source_{src}", cnt)
for split, cnt in tbl_split["count"].items():
    add_row("composition", f"samples_split_{split}", cnt)
for prov, cnt in unified["test_provenance"].value_counts(dropna=False).items():
    add_row("composition", f"test_provenance_{prov}", cnt)

# Label mapping
add_row("label_mapping", "AAAC_category=safe", "safe")
add_row("label_mapping", "AAAC_category!=safe", "adversarial")
add_row("label_mapping", "OSACT_layer1_label=safe", "safe")
add_row("label_mapping", "OSACT_layer1_label=hate_speech", "adversarial")
add_row("label_mapping", "OSACT_test_labels", "withheld (blind test set), NaN in binary_label")

# Dataset sources
add_row("sources", "AAAC_status", "published/frozen, official splits preserved")
add_row("sources", "OSACT_status", "official splits preserved (train/dev/test -> train/validation/test)")

# Duplicate analysis 
add_row("duplicates", "exact_duplicate_rows", len(exact_dupes))
add_row("duplicates", "normalized_duplicate_rows", len(norm_dupes))
add_row("duplicates", "near_duplicate_rows_cosine_ge_0.85", len(near_dupes))
add_row("duplicates", "cross_source_exact_duplicate_groups", n_cross_source_groups)
add_row("duplicates", "total_flagged_rows", len(duplicates_report))
add_row("duplicates", "total_flagged_pct_of_dataset", round(len(duplicates_report) / total_samples * 100, 2))

# Class balance
for label, cnt in unified["binary_label"].value_counts(dropna=False).items():
    add_row("class_balance", f"unified_{label}", cnt)
for label, cnt in aaac_only["binary_label"].value_counts(dropna=False).items():
    add_row("class_balance", f"AAAC_only_{label}", cnt)
for label, cnt in osact_only["binary_label"].value_counts(dropna=False).items():
    add_row("class_balance", f"OSACT_only_{label}", cnt)
add_row("class_balance", "recommendation", "class_weight=balanced (no resampling); see Part 7 for justification")

# Language statistics 
for lang, cnt in tbl_lang["count"].items():
    add_row("language_stats", f"samples_{lang}", cnt)
add_row("language_stats", "vocabulary_size_overall", vocab_overall)
add_row("language_stats", "vocabulary_size_AAAC", vocab_aaac)
add_row("language_stats", "vocabulary_size_OSACT", vocab_osact)
add_row("language_stats", "mean_token_length_overall", round(unified["token_count"].mean(), 2))

# Validation results 
for k, v in validation_report.items():
    if isinstance(v, (dict, list)):
        continue  # keep the flat report scalar-only; detailed rows already in duplicates/validation printouts
    add_row("validation", k, v)

research_report = pd.DataFrame(report_rows)
research_report.to_csv(OUTPUT_DIR / "Layer1_unified_report.csv", index=False, encoding="utf-8")
print(f"Layer1_unified_report.csv written: {len(research_report)} rows")
display(research_report)
